# 📏 Model Evaluation

Evaluate the trained model on the MVTec AD test set:
- Image-level metrics (ROC-AUC, F1, Precision, Recall)
- Pixel-level metrics (Pixel AUROC, Pixel F1, IoU)
- Visual detection samples with heatmaps
- Model comparison

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import json
from PIL import Image

from src.detect import DefectDetector
from src.evaluate import evaluate_model, evaluate_sample_visualizations
from src.dataset import NormalDataset, get_transforms
from torch.utils.data import DataLoader

DATA_DIR = '../data/mvtec_anomaly_detection/bottle'

## 1. Load Model and Calibrate Threshold

In [ ]:
detector = DefectDetector('../models/autoencoder_best.pth')

# Calibrate threshold using validation data
val_transform = get_transforms(256, augment=False)
val_dataset = NormalDataset(os.path.join(DATA_DIR, 'train', 'good'), transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

threshold = detector.calibrate_threshold(val_loader, percentile=99.0)
print(f'Calibrated threshold: {threshold:.6f}')

## 2. Evaluate on Test Set

In [ ]:
metrics = evaluate_model(
    detector=detector,
    test_dir=os.path.join(DATA_DIR, 'test'),
    ground_truth_dir=os.path.join(DATA_DIR, 'ground_truth'),
    output_dir='../artifacts/evaluation',
)

## 3. Sample Detection Visualizations

In [ ]:
# Detect on specific samples
test_dir = os.path.join(DATA_DIR, 'test')

for defect_type in sorted(os.listdir(test_dir)):
    type_dir = os.path.join(test_dir, defect_type)
    if not os.path.isdir(type_dir):
        continue
    
    images = sorted([f for f in os.listdir(type_dir) if f.endswith('.png')])[:2]
    for img_name in images:
        result = detector.detect(os.path.join(type_dir, img_name))
        
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f'{defect_type} - {result["prediction"]} (score: {result["anomaly_score"]:.4f})', fontsize=14)
        
        axes[0].imshow(result['original'])
        axes[0].set_title('Original')
        axes[0].axis('off')
        
        axes[1].imshow(result['reconstructed'])
        axes[1].set_title('Reconstructed')
        axes[1].axis('off')
        
        axes[2].imshow(result['heatmap'], cmap='jet')
        axes[2].set_title('Anomaly Heatmap')
        axes[2].axis('off')
        
        axes[3].imshow(result['overlay'])
        axes[3].set_title('Defect Localization')
        axes[3].axis('off')
        
        plt.tight_layout()
        plt.show()

## 4. Results Summary

In [ ]:
# Load saved metrics
with open('../artifacts/evaluation/metrics.json') as f:
    m = json.load(f)

print('\n' + '='*50)
print('FINAL EVALUATION RESULTS')
print('='*50)
print(f'\nImage-Level:')
print(f'  AUROC:     {m["image_auroc"]:.4f}')
print(f'  F1-Score:  {m["image_f1"]:.4f}')
print(f'  Precision: {m["image_precision"]:.4f}')
print(f'  Recall:    {m["image_recall"]:.4f}')
print(f'\nPixel-Level:')
print(f'  AUROC:     {m["pixel_auroc"]:.4f}')
print(f'  F1-Score:  {m["pixel_f1"]:.4f}')
print(f'  IoU:       {m["pixel_iou"]:.4f}')